# Notebook Overview

Purpose:

*   Transform Bronze order data into a standardized Silver dataset.
*   Validate order records against documented data-quality rules.
*   Separate accepted and quarantined records.
*   Persist outputs and reconcile quality metrics.

### Configuration / Paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

FILE_PATH = "/content/drive/MyDrive/pyspark-ecommerce-analytics-data"
BRONZE_ORDER_PATH = f"{FILE_PATH}/bronze/orders"
QUARANTINE_ORDER_PATH = f"{FILE_PATH}/quarantine/orders"
SILVER_ORDER_PATH = f"{FILE_PATH}/silver/orders"

Mounted at /content/drive


### Spark Session

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = SparkSession.builder.appName('order-silver').getOrCreate()
spark.version

'4.0.4'

### Load Data from Bronze

In [3]:
def load_data(FILE_PATH):
  try:
    df_order = spark.read.parquet(FILE_PATH).withColumn(
        '_source_file', F.lit(F.input_file_name())

    )
    print('Data Load Successful')
    return df_order

  except Exception as e:
    raise RuntimeError(f"Data Load Failed: {str(e)}")

df_order = load_data(BRONZE_ORDER_PATH)

Data Load Successful


### Row Count and Bronze Order Schema

In [4]:
def schema_and_count(df):
  print('\nRow Count:', df.count())
  print('Schema')
  print('======')
  df.printSchema()

schema_and_count(df_order)


Row Count: 30000
Schema
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- order_total: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = false)



### Validate the Bronze Schema

In [5]:
def validate_orders_schema(df):
  try:
    expected_columns = ['order_id',
                        'customer_id',
                        'order_date',
                        'order_status',
                        'payment_method',
                        'shipping_country',
                        'shipping_city',
                        'order_total',
                        '_ingestion_timestamp',
                        '_source_file'
                        ]

    df_column = df.columns

    missing_cols = list(set(expected_columns) - set(df_column))

    expected_schema = {
          'order_id': 'string',
          'customer_id': 'string',
          'order_date': 'string',
          'order_status': 'string',
          'payment_method': 'string',
          'shipping_country': 'string',
          'shipping_city': 'string',
          'order_total': 'string',
          '_ingestion_timestamp': 'timestamp',
          '_source_file': 'string'
      }

    actual_types = dict(df.dtypes)

    type_missmatch = []
    for col, expected_type in expected_schema.items():
      if col in actual_types and actual_types[col] != expected_type:
        type_missmatch.append(f"Column '{col}' expected '{expected_type}', but got '{actual_types[col]}'")

    error_messages = []
    if missing_cols:
      error_messages.append(f'Missing_cols: {missing_cols}')
    if type_missmatch:
      error_messages.append(f'Type Missmatch: {type_missmatch}')

    if error_messages:
      raise ValueError("Order Schema Validation Failed:\n" + "\n".join(error_messages))
    print("Schema Validation passed successfully!")

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'validate_orders_schema': {str(e)}") from e

### Preserve Audit Information

In [6]:
def preserve_audit(df):
  df_preserve = df
  df_preserve = df.withColumn(
      '_original_order_date',
      F.col('order_date')
      ).withColumn(
          '_original_order_total',
          F.col('order_total')
      )

  return df_preserve

### Standardize String Columns

In [7]:
def standardize_order_strings(df):
  try:
    string_cols = [
        'order_id',
        'customer_id',
        'order_date',
        'order_status',
        'payment_method',
        'shipping_country',
        'shipping_city',
        'order_total',
        '_source_file'
      ]

    df_clean = df

    for c in string_cols:
      df_clean = df_clean.withColumn(
          c,
          F.when(F.trim(F.lower(F.col(c))).isin('n/a', 'na', 'unknown', 'null', ''), None).otherwise(F.trim(F.col(c)))
      )

    return df_clean
  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'standardize_order_strings': {str(e)}") from e

### Cast Types

In [8]:
from pyspark.sql.types import DecimalType, DateType

def cast_order_types(df):
  try:
    if 'order_total' in df.columns:
      df_typed = df.withColumn('order_total', F.col('order_total').try_cast(DecimalType(10,2)))

    if 'order_date' in df.columns:
      df_typed = df_typed.withColumn('order_date', F.col('order_date').try_cast(DateType()))

    return df_typed

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'cast_order_types': {str(e)}") from e

### Validate Order Records

In [9]:
def validate_order_records(df):
  try:
    valid_status = ['returned', 'shipped', 'processing', 'completed', 'cancelled', 'in transit']
    valid_payment_method = ['bank transfer', 'crypto', 'paypal', 'card', 'wallet', 'upi']
    df_validated = df.withColumn(
        'is_valid',
        F.when(F.col('order_id').isNull(), 'Missing or NULL Order ID')
        .when(F.col('customer_id').isNull(), 'Missing or NULL Customer ID')
        .when((F.col('order_date').isNull()) & (F.col('_original_order_date').isNotNull()), 'Invalid Order Date')
        .when((F.col('order_status').isNull()) | (~F.lower(F.col('order_status')).isin(valid_status)), 'Missing or Invalid Order Status')
        .when((F.col('payment_method').isNull()) | (~F.lower(F.col('payment_method')).isin(valid_payment_method)), 'Missing or Invalid Payment Method')
        .when((F.col('order_total').isNull()) | (F.col('order_total') == 0) | (F.col('order_total') < 0), 'Invalid or Missing Order Total')
    )

    return df_validated

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'validate_customer_records': {str(e)}") from e


### Handle Duplicate Order IDs

In [10]:
from pyspark.sql import Window as W

def handle_duplicate_order_ids(df):
  try:
    window_spec = W.partitionBy('order_id').orderBy(F.col('_ingestion_timestamp').desc())

    df_ranked = df.withColumn(
        'rank',
        F.row_number().over(window_spec)
    )

    df_mark = df_ranked.withColumn(
        'is_valid',
        F.coalesce(
            F.col('is_valid'),
            F.when(F.col('rank') > 1, 'Duplicate Customer ID')
        )
    ).drop('rank')

    return df_mark

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'handle_duplicate_order_ids': {str(e)}") from e

### Split Valid and Quarantined Records

In [11]:
def split_orders_by_quality(df):
  try:
    df_valid = df.filter(F.col('is_valid').isNull()).drop('is_valid', '_original_signup_date')

    df_quarantine = df.filter(F.col('is_valid').isNotNull())

    return df_valid, df_quarantine

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Unexpected failure in 'split_orders_by_quality': {str(e)}") from e

### Orchestrate the Order Transformation

In [12]:
def transform_orders(df):
  try:
    print('===============================')
    print('TRANSFORMING DATA')
    print('===============================')
    print('Start: Validating Bronze Schema')
    validate_orders_schema(df)
    print('Finished: Validaing Bronze Schema')
    print('Start: Preserving Data for Audit')
    preserved_df = preserve_audit(df)
    print('Finished: Preserving Data for Audit')
    print('Start: Standardizing String Columns')
    standardize_df = standardize_order_strings(preserved_df)
    print('Finished: Standardizing String Columns')
    print('Start: Cast type Column')
    casted_df = cast_order_types(standardize_df)
    print('Finished: Cast type Column')
    print('Start: Validating Orders Record')
    validated_df = validate_order_records(casted_df)
    print('Finished: Validating Orders Record')
    print('Start: Handling and Marking duplicate id')
    df_mark = handle_duplicate_order_ids(validated_df)
    print('Finished: Handling and Marking duplicate id')
    print('Spiliting and Returning Validated and Quanrantine data')
    print('===============================')
    print('TRANSFORMING COMPLETE')
    print('===============================')
    return split_orders_by_quality(df_mark)

  except Exception as e:
    if isinstance(e, ValueError):
      raise e
    raise RuntimeError(f"[Pipeline Error] Failed inside 'transform_orders': {str(e)}") from e

### **Transforming Order Data**

In [13]:
valid_df, quarantine_df = transform_orders(df_order)

TRANSFORMING DATA
Start: Validating Bronze Schema
Schema Validation passed successfully!
Finished: Validaing Bronze Schema
Start: Preserving Data for Audit
Finished: Preserving Data for Audit
Start: Standardizing String Columns
Finished: Standardizing String Columns
Start: Cast type Column
Finished: Cast type Column
Start: Validating Orders Record
Finished: Validating Orders Record
Start: Handling and Marking duplicate id
Finished: Handling and Marking duplicate id
Spiliting and Returning Validated and Quanrantine data
TRANSFORMING COMPLETE


### Calculate quality metrics and reconcile

In [14]:
def generate_pipeline_metrics(df_bronze, df_valid, df_quarantine):
    print("\n" + "=" * 65)
    print(" PIPELINE RECONCILIATION & QUALITY METRICS REPORT")
    print("=" * 65)

    input_count = df_bronze.count()
    silver_count = df_valid.count()
    quarantine_count = df_quarantine.count()

    print("\n[1] RECONCILIATION SUMMARY")
    print(f"  • Total Bronze Input Count      : {input_count:,}")
    print(f"  • Accepted Silver Count         : {silver_count:,}")
    print(f"  • Quarantined Count             : {quarantine_count:,}")

    sum_outputs = silver_count + quarantine_count
    is_balanced = (input_count == sum_outputs)
    print(f"  • Reconciliation Balance Check  : {'PASSED (Match)' if is_balanced else 'FAILED (Mismatch!)'}")
    if not is_balanced:
        print(f"    WARNING: Input ({input_count:,}) != Silver ({silver_count:,}) + Quarantine ({quarantine_count:,})")

    print("\n[2] QUARANTINE FAILURE BREAKDOWN (Validation & Duplicates)")
    if quarantine_count > 0:
        print("  Breakdown of reasons records were quarantined:")
        df_quarantine.groupBy("is_valid").count().orderBy(F.col("count").desc()).show(truncate=False)
    else:
        print("  • No quarantined records found. 100% clean dataset!")

    print("\n[3] NULL COUNTS IN SILVER LAYER")
    print("  (Tracking optional profile fields alongside required ones)")
    null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_valid.columns]
    df_valid.select(null_exprs).show(truncate=False)

    print("\n[4] UNIQUENESS CHECK (order_id in Silver)")
    distinct_orders = df_valid.select("order_id").distinct().count()
    is_unique = (silver_count == distinct_orders)
    print(f"  • Total Silver Rows             : {silver_count:,}")
    print(f"  • Unique Order IDs           : {distinct_orders:,}")
    print(f"  • Uniqueness Constraint         : {'PASSED (No duplicate IDs in Silver)' if is_unique else 'FAILED (Duplicates present!)'}")

    print("=" * 65)

generate_pipeline_metrics(df_order, valid_df, quarantine_df)


 PIPELINE RECONCILIATION & QUALITY METRICS REPORT

[1] RECONCILIATION SUMMARY
  • Total Bronze Input Count      : 30,000
  • Accepted Silver Count         : 29,996
  • Quarantined Count             : 4
  • Reconciliation Balance Check  : PASSED (Match)

[2] QUARANTINE FAILURE BREAKDOWN (Validation & Duplicates)
  Breakdown of reasons records were quarantined:
+------------------------------+-----+
|is_valid                      |count|
+------------------------------+-----+
|Invalid or Missing Order Total|2    |
|Invalid Order Date            |1    |
|Duplicate Customer ID         |1    |
+------------------------------+-----+


[3] NULL COUNTS IN SILVER LAYER
  (Tracking optional profile fields alongside required ones)
+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+--------------------+------------+--------------------+---------------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_country|shipping_

### Adding Metadata and Enforcing Schema

In [15]:
from pyspark.sql.types import TimestampType


# Adding metadata to Quarantine Dataset
quarantine_df = quarantine_df.withColumn('_silver_processed_timestamp', F.current_timestamp())

# Adding metadata to Valid Dataset
valid_df = valid_df.withColumn('_silver_processed_timestamp', F.current_timestamp())

silver_order_df = valid_df.select(
    'order_id',
    'customer_id',
    'order_date',
    'order_status',
    'payment_method',
    'shipping_country',
    'shipping_city',
    'order_total',
    '_silver_processed_timestamp',
    '_source_file'
)

silver_order_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- order_total: decimal(10,2) (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = false)
 |-- _source_file: string (nullable = true)



### ***Write into Silver***

In [16]:
try:
  print('INFO: Writing records to Quarantine layer...')
  # Write Quarantine Data
  quarantine_df.write.mode('overwrite').parquet(QUARANTINE_ORDER_PATH)
  print('SUCCESS: Quarantine dataset written successfully.')

  print('INFO: Writing clean records to Silver layer...')
  # Write Siver Data into Layer
  silver_order_df.write.mode('overwrite').parquet(SILVER_ORDER_PATH)
  print('SUCCESS: Silver dataset written successfully.')

except Exception as e:
    raise RuntimeError(f"[Pipeline Error] Failed to write Silver dataset: {str(e)}") from e


INFO: Writing records to Quarantine layer...
SUCCESS: Quarantine dataset written successfully.
INFO: Writing clean records to Silver layer...
SUCCESS: Silver dataset written successfully.


### Read Back and Verify

In [17]:
df = spark.read.parquet(SILVER_ORDER_PATH)

quarantine_df = spark.read.parquet(QUARANTINE_ORDER_PATH)

schema_and_count(df)
schema_and_count(quarantine_df)


Row Count: 29996
Schema
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- order_total: decimal(10,2) (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)


Row Count: 4
Schema
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- order_total: decimal(10,2) (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _original_order_date: